# Redis Pub/Sub Examples

Redis Pub/Sub implements a messaging paradigm where senders (publishers) send messages to channels without knowledge of receivers (subscribers). This is useful for real-time notifications, event-driven architectures, and decoupled communication between services.

## Start off by connecting to the Redis server

Pub/Sub requires two connections: one for the subscriber (which blocks waiting for messages) and one for the publisher.

In [ ]:
import redis

# Subscriber connection
sub_client = redis.Redis(decode_responses=True)
sub_client.ping()

In [ ]:
# Publisher connection (must be separate from subscriber)
pub_client = redis.Redis(decode_responses=True)
pub_client.ping()

## Basic Pub/Sub

Subscribe to a channel, then publish a message to it.

In [ ]:
# Create a PubSub object and subscribe to a channel
pubsub = sub_client.pubsub()
pubsub.subscribe("notifications")

# The subscribe call returns a confirmation message
message = pubsub.get_message()
print("Subscribe confirmation:", message)

In [ ]:
# Publish a message to the channel
pub_client.publish("notifications", "Hello from Redis Pub/Sub!")

In [ ]:
# Receive the published message
message = pubsub.get_message()
print("Received message:", message)

## Pattern-Based Subscriptions

You can subscribe to multiple channels matching a pattern using `psubscribe`. The `*` wildcard matches any characters.

In [ ]:
pubsub_pattern = sub_client.pubsub()
pubsub_pattern.psubscribe("user:*:events")

# Confirm subscription
pubsub_pattern.get_message()

In [ ]:
# Publish to channels matching the pattern
pub_client.publish("user:123:events", "login")
pub_client.publish("user:456:events", "purchase")

# Receive messages
msg1 = pubsub_pattern.get_message()
msg2 = pubsub_pattern.get_message()
print("Message 1:", msg1)
print("Message 2:", msg2)

## Listening in a Loop

In practice, you typically listen for messages in a loop using `listen()` or `get_message()` with a timeout.

In [ ]:
pubsub_loop = sub_client.pubsub()
pubsub_loop.subscribe("alerts")

# Consume the subscription confirmation
pubsub_loop.get_message()

# Publish a few messages
pub_client.publish("alerts", "System starting up")
pub_client.publish("alerts", "All services healthy")
pub_client.publish("alerts", "Scheduled maintenance in 30 minutes")

In [ ]:
# Read all pending messages with a timeout
while True:
    message = pubsub_loop.get_message(timeout=1.0)
    if message is None:
        break
    if message["type"] == "message":
        print(f"Alert: {message['data']}")

## Unsubscribing

Always unsubscribe when done to clean up resources.

In [ ]:
# Unsubscribe from specific channels
pubsub.unsubscribe("notifications")
pubsub_loop.unsubscribe("alerts")
pubsub_pattern.punsubscribe("user:*:events")

# Close the PubSub objects
pubsub.close()
pubsub_loop.close()
pubsub_pattern.close()

## Pub/Sub with JSON Messages

Pub/Sub messages are strings, so you can easily send JSON-encoded data for structured communication.

In [ ]:
import json

pubsub_json = sub_client.pubsub()
pubsub_json.subscribe("task_updates")
pubsub_json.get_message()  # consume confirmation

In [ ]:
# Publish a JSON-encoded message
task = {
    "task_id": "abc-123",
    "status": "completed",
    "result": {"rows_affected": 42}
}
pub_client.publish("task_updates", json.dumps(task))

In [ ]:
# Receive and decode the JSON message
message = pubsub_json.get_message()
if message and message["type"] == "message":
    data = json.loads(message["data"])
    print(f"Task {data['task_id']} is {data['status']}")
    print(f"Result: {data['result']}")

In [ ]:
# Cleanup
pubsub_json.unsubscribe("task_updates")
pubsub_json.close()